# Block 2 — Clinical Knowledge Graph & Prior Validation Engine
**Medical Document Intelligence System**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block2/block2/Block_2_Medical_Knowledge_Graph.ipynb)

---
### Overview
Block 2 provides a **frozen, deterministic clinical knowledge graph and prior engine**. It acts as the medical ground truth for the document understanding pipeline:
1. **Catalog & Profile Bundles:** Maps 138+ checkbox items and health check plans (Lipid, Liver, Renal, Anemia, Diabetes) to component tests.
2. **Tube Constraints:** Computes exact laboratory specimen tube requirements (`EDTA`, `CB`, `Fl`, `Cit`, `Urine`, `Stool`, `pap`, `UBT`).
3. **Clinical Lexicon & Alias Resolution:** Normalizes clinical abbreviations, acronyms, and misspellings (`SGPT` → `ALT`, `HbA1c` → `Glycated Hemoglobin`).
4. **Fuzzy Write-in Matching:** Resolves messy doctor handwriting in *OTHERS* fields against known clinical tests.
5. **Bayesian / Prior Ranking (`assume()`):** Biases downstream HTR candidate selection based on already-checked tests.
6. **Cross-Field Validation:** Detects tube shortages, redundant test orders, and generates high-confidence audit reports.

## 1. Setup & Environment
Install minimal requirements (Pydantic) and import `med_doc.kg`.

In [ ]:
# In Google Colab, clone repo if running in a fresh session:
import os, sys
if not os.path.exists('med_doc'):
    !git clone -b block2 https://github.com/RwaRwa599/epq3.git repo
    if os.path.exists('repo/block2'):
        sys.path.insert(0, 'repo/block2')
        os.chdir('repo/block2')
    else:
        sys.path.insert(0, 'repo')
        os.chdir('repo')

!pip install -q "pydantic>=2.0.0" "matplotlib>=3.7.0"

from med_doc.kg import KnowledgeGraph, CatalogueItem, RankedCandidate, ValidationResult
print("✓ Environment initialized and med_doc.kg imported successfully!")

## 2. Load the Frozen Clinical Knowledge Graph
Load the verified knowledge base (`v1` covering 138 printed test checkboxes + common write-ins).

In [ ]:
kg = KnowledgeGraph.load()
print(f"[+] Loaded Knowledge Graph Version: {kg.version}")
print(f"[+] Total Catalogue Test Items:    {len(kg.catalogue)}")
print(f"[+] Total Profile Bundles:         {len(kg.profile_bundles)}")
print(f"[+] Total Tube Mapping Rules:       {len(kg.test_to_tube)}")
print(f"[!] Disclaimer: {kg.disclaimer}")

## 3. Health Check Profile & Bundle Expansion
When a clinician ticks a composite profile (e.g. `Lipid Profile`, `Liver Function Test`, `Anemia Profile`), the KG expands it to its atomic clinical component tests.

In [ ]:
profiles_to_inspect = ["profile_lipid", "profile_liver", "profile_diabetes", "profile_anemia", "profile_renal"]

for pid in profiles_to_inspect:
    item = kg.get_item(pid)
    label = item.label if item else pid
    components = kg.expand_profile(pid)
    comp_labels = [kg.get_item(c).label if kg.get_item(c) else c for c in components]
    print(f"• {label} ({pid}):")
    print(f"    → Components ({len(components)}): {', '.join(comp_labels)}\n")

## 4. Laboratory Specimen & Tube Calculation
Calculate the exact physical blood collection tubes and specimen containers required for any combination of ordered tests.

In [ ]:
order_scenarios = [
    {"name": "Routine Check", "tests": ["cbc", "alt", "glucose_fasting"]},
    {"name": "Cardiovascular & Metabolic", "tests": ["profile_lipid", "hba1c", "profile_renal"]},
    {"name": "Antenatal & Screening", "tests": ["profile_antenatal", "pap_smear", "urinalysis"]},
    {"name": "Gastric & Stool Panel", "tests": ["urea_breath_test", "stool_occult_blood"]},
]

for sc in order_scenarios:
    tubes = kg.calculate_expected_tubes(sc["tests"])
    print(f"Scenario: {sc['name']}")
    print(f"  Ordered IDs: {sc['tests']}")
    print(f"  → Required Tubes: {tubes}")
    print("-" * 60)

## 5. Clinical Alias & Handwriting Resolution
Resolve shorthand doctor acronyms and fuzzy match unconstrained handwriting write-in fields (*OTHERS*).

In [ ]:
# Test Alias Resolution
acronyms = ["SGPT", "ALT", "SGOT", "AST", "HbA1c", "VDRL", "RPR", "TSH", "FT4", "CBC"]
print("=== ACRONYM / ALIAS RESOLUTION ===")
for ac in acronyms:
    fid = kg.resolve_alias(ac)
    item = kg.get_item(fid) if fid else None
    label = item.label if item else "Unknown"
    print(f"  '{ac:7s}' → Field ID: {fid:16s} (Canonical: {label})")

# Test Fuzzy Handwriting Resolution for Write-Ins (OTHERS)
print("\n=== FUZZY HANDWRITING WRITE-IN RESOLUTION ===")
handwriting_samples = ["cultur and sensitivty", "growth hormne", "fasting glucose", "covid igg antibody"]
for hw in handwriting_samples:
    matches = kg.fuzzy_match_catalogue(hw, top_k=2)
    print(f"\n  Input Handwriting: '{hw}'")
    for rank, m in enumerate(matches, 1):
        print(f"    [{rank}] {m.value:30s} | Score: {m.score:.3f} | Tier: {m.tier} | Reason: {m.reason}")

## 6. Contextual Prior Ranking (`assume()`)
Use Bayesian / medical priors to boost handwriting recognition candidates when related tests or profiles are already checked on the sheet.

In [ ]:
# Scenario: Doctor checked 'profile_lipid', wrote messy 'triglycerides' in OTHERS
context = {"ticked_ids": ["profile_lipid"]}
candidates = kg.assume("others", "triglyceride", context=context, top_k=3)

print("Context: ['profile_lipid'] is already checked.")
print("assume('others', 'triglyceride'):")
for c in candidates:
    print(f"  → {c.value:25s} Score: {c.score:.3f} (Tier {c.tier}) - {c.reason}")

## 7. Cross-Field Clinical Validation Engine
Simulate full request validation, detecting physical tube shortages and redundant test ordering.

In [ ]:
# Case 1: Complete Valid Request
report_valid = kg.validate_request(
    ticked_ids=["cbc", "alt", "glucose_fasting"],
    observed_tubes={"EDTA": 1, "CB": 1, "Fl": 1}
)
print("CASE 1 (Valid Order):")
print(f"  Valid: {report_valid.is_valid} | Confidence: {report_valid.confidence}")
print(f"  Expected Tubes: {report_valid.expected_tubes}")
print(f"  Observed Tubes: {report_valid.observed_tubes}")
print(f"  Discrepancies:  {report_valid.discrepancies}")
print(f"  Warnings:       {report_valid.warnings}\n")

# Case 2: Tube Shortage Discrepancy
report_shortage = kg.validate_request(
    ticked_ids=["cbc", "alt", "glucose_fasting"],
    observed_tubes={"EDTA": 1, "CB": 0, "Fl": 1}  # Missing Clotted Blood tube!
)
print("CASE 2 (Tube Shortage):")
print(f"  Valid: {report_shortage.is_valid} | Confidence: {report_shortage.confidence}")
print(f"  Discrepancies:  {report_shortage.discrepancies}")
print(f"  Warnings:       {report_shortage.warnings}\n")

# Case 3: Redundant Test Selection
report_redundant = kg.validate_request(
    ticked_ids=["profile_lipid", "chol_total"],  # Total Cholesterol is already inside Lipid Profile
    observed_tubes={"CB": 1}
)
print("CASE 3 (Redundant Selection):")
print(f"  Valid: {report_redundant.is_valid} | Confidence: {report_redundant.confidence}")
print(f"  Discrepancies:  {report_redundant.discrepancies}")
print(f"  Warnings:       {report_redundant.warnings}")

## 8. Export Structured Clinical Audit JSON
Generate standardized JSON output ready for downstream LIS systems or downstream Block 3/4 pipelines.

In [ ]:
import json

output_json = report_valid.model_dump_json(indent=2)
print("=== EXPORTED VALIDATION JSON ===")
print(output_json)

with open("validation_report.json", "w") as f:
    f.write(output_json)
print("\n[+] Successfully saved validation_report.json!")